# N125 · 位掩码、异或与二进制Trie

**目标：** 把集合操作和按位最优选择转换成整数操作。

**先修：** N124, N077, N015。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 子集/子掩码；XOR消去；位分治；二进制Trie；线性基扩展。

**配套讲解来源：** [同名逐章意见](../../comment/125_bitmasks_xor_trie.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这章把三件看似不相干的事，统一成"整数的二进制位就是数据结构"这一个思想：

1. 枚举集合的所有子集。把集合编码成整数掩码（第 i 位是 1 表示元素 i 在集合里），例如 mask = 0b10110 = 22 代表集合 {1,2,4}（第 1、2、4 位是 1）。它一共有 2³ = 8 个子集：22, 20, 18, 16, 6, 4, 2, 0。问题是怎么按降序一个个吐出来，不用递归、不用额外集合。
2. 找"只出现一次的数"。数组 [2,2,1] 里别的数都成对出现，只有一个落单，答案是 1。用异或的自消性质，一次遍历就能把它揪出来。
3. 数组中任取两个数，异或值最大能到多少？例如 nums = [3,10,5,25,2,8]，答案是 28，因为 5 ^ 25 = 00101 ^ 11001 = 11100 = 28。暴力要试全部 O(n²) 个组合；二进制 Trie 让每个数只花 O(位数) 就找到自己的"最佳搭档"。

这三件事的共同底层是：按位看问题，高一位的权重（2^k）比所有低位加起来（2^k − 1）还大，所以按位决策可以做贪心。

## 2. 基础知识：先读懂这些词

- **位掩码（bitmask）表示集合**：用一个整数的第 i 个二进制位当"元素 i 在不在集合里"的开关。集合 {1,2,4} 就是 2¹+2²+2⁴ = 22。集合的交就是 &、并就是 |、差可以用 & ~。
- **子掩码（submask）**：一个掩码 sub 是 mask 的子掩码，当且仅当 sub 的每个 1 都出现在 mask 的对应位上（数学判断就是 sub & mask == sub）。子掩码恰好对应原集合的所有子集。
- **(sub−1) & mask 迭代**：枚举子掩码的循环技巧。sub 减一会把最低的 1 借位拆开，再与 mask 相与就把"借出来的不属于 mask 的位"全部清掉，结果正好是"比 sub 小的下一个合法子掩码"。如此循环可以从 mask 一路降到 0，且严格降序、不会重复。
- **异或（XOR，^）与自消**：异或的规则是两位不同为 1。两个关键性质：x ^ x = 0（自己消自己）、x ^ 0 = x。所以一堆数异或起来，出现偶数次的数两两抵消，最后只剩下出现奇数次的那部分。
- **二进制 Trie（01-Trie）**：一棵只有 0、1 两种出边的字典树，把每个整数按二进制从最高位到最低位作为路径插进去。树上从根到叶的一条路径就是一个数。它把"一组数"变成"一张按位组织的地图"，特别适合回答"和 x 异或谁能最大"这类按位问题。
- **高位优先贪心**：第 k 位的权重是 2^k，而它下面所有位全开也只有 2^k − 1。所以判断"异或结果能不能大"时，高位能不能取 1 是一票否决的：高位取 1 的任何方案都压过高位取 0 的一切方案，低位根本没资格补偿。
- **位分治**：把一个按位的问题拆成"最高位怎么选 + 剩下位递归"，Trie 的每一层天然就是这个分治的一层。
- **bit_length**：Python 方法，返回该非负整数二进制去掉前导零后的位数，比如 25 的二进制是 11001，bit_length 是 5。本章用它自动决定 Trie 的位数。
- **线性基（扩展知识）**：知识点清单里点名的扩展方向——把一组数的"异或张成空间"压缩成至多 B 个基底，也能解最大异或子集等问题；本章不实现，知道有这条延展路即可。

## 3. 思路推导：从小例子开始

- Step 1：先手算子掩码枚举。mask = 0b10110 = 22。从 sub=22 开始，反复做 sub = (sub−1) & mask：
  - 22−1=21 (10101)，& 10110 = 10100 = 20；
  - 20−1=19 (10011)，& 10110 = 10010 = 18；
  - 18−1=17 (10001)，& 10110 = 10000 = 16；
  - 16−1=15 (01111)，& 10110 = 00110 = 6（注意这一步大跳：16 的低位没有 1 可借，减法借穿了，掩码一把拽回来）；
  - 6−1=5 (00101)，& 10110 = 00100 = 4；
  - 4−1=3 (00011)，& 10110 = 00010 = 2；
  - 2−1=1 (00001)，& 10110 = 00000 = 0；到 0 收工。
  完整序列 [22, 20, 18, 16, 6, 4, 2, 0]，正好是全部 8 个子集的降序排列。notebook “运行示例”部分 的表就是把这条链连同二进制和元素个数打印出来：

| 子掩码 | 二进制 | 元素数 |
|---|---|---|
| 22 | 10110 | 3 |
| 20 | 10100 | 2 |
| 18 | 10010 | 2 |
| 16 | 10000 | 1 |
| 6 | 00110 | 2 |
| 4 | 00100 | 1 |
| 2 | 00010 | 1 |
| 0 | 00000 | 0 |

- Step 2：为什么 (sub−1)&mask 是"下一个更小的子掩码"？sub−1 一定比 sub 小，但它可能借出一些 mask 没有的位（不合法）；与 mask 相与后，mask 外的位全被清零，得到的是"不超过 sub−1 的、由 mask 的位组成的最大编码"。既是合法子掩码、又严格小于 sub，而且不跳过任何候选——所以序列无重复、无遗漏、严格降序。特别注意 0 之后如果继续算会得到 (0−1)&mask = mask，绕回开头，所以代码在 0 处显式返回。
- Step 3：异或找落单数。把 [2,2,1] 依次异或：0^2=2，2^2=0，0^1=1。成对的 2 在第二轮自消归零，最后剩下 1。只要"除答案外每个数都出现偶数次"，这个一行循环就是完整题解。
- Step 4：Trie 上找最大异或。把 [3,10,5,25,2,8] 全部插入 5 位 Trie（25 的 bit_length 是 5），然后对每个 x 沿树走：每一层优先走"x 这一位取反"的分支（异或该位才能得 1），走不通才走同位分支。以 x=5 (00101) 为例逐层选反位，最终锁定搭档 25 (11001)，异或 11100 = 28。贪心为什么对：最高位若能取 1，收益 2⁴ = 16 已经超过后面四位全取 1 的 15，所以每层"能取反就取反"必然全局最优。
- Step 5：小位宽演示（对应测试里的 BinaryTrie(4)）。插入 3 (0011) 两次后查 max_xor(12)（12 = 1100）：
  - 第 3 层（权 8）：12 的位是 1，找 0 分支——存在（3 的路径以 0 开头），答案记 8；
  - 第 2 层（权 4）：位是 1，找 0 分支——存在，答案累加到 12；
  - 第 1 层（权 2）：位是 0，找 1 分支——存在，答案累加到 14；
  - 第 0 层（权 1）：位是 0，找 1 分支——存在，答案累加到 15。
  结果 15 = 12 ^ 3，正是 12 与 Trie 中唯一的数 3 的异或。

## 4. 核心代码：enumerate_submasks

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def enumerate_submasks(mask):
    if mask < 0:
        raise ValueError('nonnegative mask required')
    out = []
    sub = mask
    while True:
        out.append(sub)
        if sub == 0:
            return out
        sub = sub - 1 & mask
```

### 逐段读懂代码

### 4.1 enumerate_submasks：子掩码降序枚举

```python
def enumerate_submasks(mask):
    if mask < 0:
        raise ValueError('nonnegative mask required')
    out = []
    sub = mask
    while True:
        out.append(sub)
        if sub == 0:
            return out
        sub = sub - 1 & mask
```

- `if mask<0: raise ValueError(...)` 挡负数：负数在 Python 里是无限补码，(sub−1)&mask 的语义完全乱套，必须拒收。
- `sub=mask` 从全集出发。循环体先收集当前子掩码再更新，顺序保证 mask 本身第一个被输出。
- `if sub==0: return out` 是关键的出口：0 也是合法子集（空集），必须先收进来；但若对 0 再做 (0−1)&mask 会得到 mask，序列绕圈，所以到 0 就停。

### 4.2 single_number：异或抵消

```python
def single_number(nums):
    answer = 0
    for x in nums:
        answer ^= x
    return answer
```

- 四行标准写法：初值 0（0 ^ x = x，不干扰结果），逐个异或。成对出现的数中途归零，最终只剩出现奇数次的值。前提（“正确性与复杂度”部分 明确写了）：除目标外其他元素都出现偶数次。

### 4.3 BinaryTrie：01 字典树

```python
class BinaryTrie:

    def __init__(self, bits=32):
        if bits < 1:
            raise ValueError('positive bit width required')
        self.bits = bits
        self.children = [[-1, -1]]
        self.size = 0

    def _check(self, x):
        if not 0 <= x < 1 << self.bits:
            raise ValueError('value outside bit width')
```

- `children` 是"节点 → [0 号孩子, 1 号孩子]"的列表，−1 表示这条边不存在。根节点 0 初始没有任何孩子。
- `if bits<1` 挡住零/负位宽；`_check` 挡住超出位宽的值（比如 4 位 Trie 装 16 会报错），把"值域契约"写死在接口里。

```python
def insert(self,x):
        self._check(x); node=0
        for shift in range(self.bits-1,-1,-1):
            bit=x>>shift&1
            if self.children[node][bit]==-1: self.children[node][bit]=len(self.children); self.children.append([-1,-1])
            node=self.children[node][bit]
        self.size+=1
```

- `for shift in range(self.bits-1,-1,-1)` 从最高位（第 bits−1 位）走到第 0 位，符合"高位优先"的世界观。
- `bit=x>>shift&1` 是紧凑写法：先把 x 右移 shift 位让目标位落到个位，再 &1 取出它。
- 没有孩子就现建一个（`len(self.children)` 正好是新节点的下标），然后 node 前进。整条路径走完，一个数就存好了。size 记录插入个数，供空树检查用。

```python
def max_xor(self,x):
        self._check(x)
        if not self.size: raise ValueError('empty trie')
        node=answer=0
        for shift in range(self.bits-1,-1,-1):
            bit=x>>shift&1; opposite=bit^1
            if self.children[node][opposite]!=-1: answer|=1<<shift; node=self.children[node][opposite]
            else: node=self.children[node][bit]
        return answer
```

- `if not self.size: raise ValueError('empty trie')` 拒绝空树查询——树上没数时谈"最大异或"没有意义。
- 每层先算 `opposite=bit^1`（把 0 变 1、把 1 变 0）：走反位意味着这一位异或结果是 1，`answer|=1<<shift` 把该位的收益记上；反位不存在才委屈走同位（该位异或结果 0，不记账）。答案的每一位就是这样逐层攒出来的。

### 4.4 find_maximum_xor：建树 + 全员查询取最大

```python
def find_maximum_xor(nums):
    if not nums:
        return 0
    trie = BinaryTrie(max(1, max(nums).bit_length()))
    for x in nums:
        trie.insert(x)
    return max((trie.max_xor(x) for x in nums))
```

- `if not nums: return 0` 处理空数组边界。
- 位宽取 `max(1, max(nums).bit_length())`：最长的数多少位，树就建多少层；max(1,·) 防止全 0 数组时 bit_length 为 0 建出零层树。
- 全部插入后，对每个 x 问一次 max_xor，取最大者。注意每个 x 查询时树里也包含它自己（自己与自己异或得 0，通常不会是最大，不影响正确性）。

## 5. 分段实现：按顺序运行

**本章接口：** `enumerate_submasks(mask)`、`single_number(nums)`、`BinaryTrie`、`find_maximum_xor(nums)`

### enumerate_submasks

In [1]:
def enumerate_submasks(mask):
    if mask < 0:
        raise ValueError('nonnegative mask required')
    out = []
    sub = mask
    while True:
        out.append(sub)
        if sub == 0:
            return out
        sub = sub - 1 & mask

### single_number

In [2]:
def single_number(nums):
    answer = 0
    for x in nums:
        answer ^= x
    return answer

### BinaryTrie

In [3]:
class BinaryTrie:

    def __init__(self, bits=32):
        if bits < 1:
            raise ValueError('positive bit width required')
        self.bits = bits
        self.children = [[-1, -1]]
        self.size = 0

    def _check(self, x):
        if not 0 <= x < 1 << self.bits:
            raise ValueError('value outside bit width')

    def insert(self, x):
        self._check(x)
        node = 0
        for shift in range(self.bits - 1, -1, -1):
            bit = x >> shift & 1
            if self.children[node][bit] == -1:
                self.children[node][bit] = len(self.children)
                self.children.append([-1, -1])
            node = self.children[node][bit]
        self.size += 1

    def max_xor(self, x):
        self._check(x)
        if not self.size:
            raise ValueError('empty trie')
        node = answer = 0
        for shift in range(self.bits - 1, -1, -1):
            bit = x >> shift & 1
            opposite = bit ^ 1
            if self.children[node][opposite] != -1:
                answer |= 1 << shift
                node = self.children[node][opposite]
            else:
                node = self.children[node][bit]
        return answer

### find_maximum_xor

In [4]:
def find_maximum_xor(nums):
    if not nums:
        return 0
    trie = BinaryTrie(max(1, max(nums).bit_length()))
    for x in nums:
        trie.insert(x)
    return max((trie.max_xor(x) for x in nums))

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [5]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

mask=0b10110
show_table(['子掩码','二进制','元素数'],[(s,format(s,'05b'),s.bit_count()) for s in enumerate_submasks(mask)])
print('最大异或',find_maximum_xor([3,10,5,25,2,8]))

子掩码,二进制,元素数
22,10110,3
20,10100,2
18,10010,2
16,10000,1
6,00110,2
4,00100,1
2,00010,1
0,00000,0


最大异或 28


## 7. 正确性、适用条件与复杂度

子掩码更新给出当前集合编码以下的最大合法编码，因此无重无漏，0必须显式处理否则会回到mask。Trie每一层选可行的最大异或位；若高位可取1，任何高位取0的方案都不可能被低位补偿。

**代价：** 含k个1的掩码有2^k个子集，输出本身就是指数规模。Trie插入/查询O(B)，总O(nB)时间空间上界。最大异或输入为非负整数；单独异或抵消要求除目标外其他元素出现偶数次。

### 展开理解

子掩码枚举为什么对：“正确性与复杂度”部分 的论证翻译过来是——(sub−1)&mask 给出的是"当前编码以下、仍然是 mask 子集的最大编码"。它不会重复（序列严格递减），也不会跳过任何子集（若真有个子集夹在 sub 和 (sub−1)&mask 之间，它既要比 (sub−1)&mask 大又要小于 sub，这与"最大"矛盾）。所以从 mask 走到 0，每个子集恰好被访问一次。0 必须显式收尾，否则下一步 (0−1)&mask = mask 会绕回起点。

Trie 贪心为什么对：某一位能不能让异或结果取 1，取决于"当前节点有没有反位孩子"，这个判断逐层独立。而高位取 1 的价值是 2^k，低位全部取 1 加起来也只有 2^k − 1，所以任何一层"明明能取 1 却取 0"的方案，都必然劣于取 1 的方案，贪心逐层取反位就是全局最优。又因为每个数插入的路径唯一，对每个 x 的查询覆盖了所有配对可能，取最大即答案。

复杂度（结合具体规模感受）：

- 子掩码枚举：含 k 个 1 的 mask 有 2^k 个子集，输出本身就是指数规模——mask=22 只有 3 个 1，8 个子集；一个 20 个 1 的 mask 就有约一百万个子集，这不是算法慢，是答案多。每个子集 O(1) 更新，生成成本与输出规模同阶。
- single_number：一次线性扫描，n=10⁶ 也只是一百万次异或，一瞬间。
- Trie：插入一个数走 B 层、查询一个数走 B 层，全程 O(nB) 时间和空间。B=32、n=2×10⁴（LC421 的规模）时约 64 万次节点操作，轻松跑完。find_maximum_xor 的 B 取自最大数的 bit_length，值域 32 位以内时 B ≤ 32。

前提别忘：本章的异或消去只适用于"其他元素都出现偶数次"；Trie 的 _check 契约要求 0 ≤ x < 2^bits；最大异或的输入必须是非负整数。

## 8. 单元测试：每个用例在检查什么

```python
assert single_number([2, 2, 1]) == 1
```
这条测异或自消的正常路径：一对 2 中途抵消，落单的 1 留到最后。

```python
assert find_maximum_xor([3, 10, 5, 25, 2, 8]) == 28
```
这条是 LC421 的官方示例：最优配对是 5 ^ 25 = 28。它同时考到位宽自适应（25 需要 5 位树）和高位贪心的正确性。

```python
for mask in range(64):
    subs = enumerate_submasks(mask)
    assert subs == sorted((s for s in range(mask + 1) if s & mask == s), reverse=True)
    assert len(subs) == len(set(subs)) == 1 << mask.bit_count()
```
这段把 mask 0..63 全查一遍：子掩码列表必须严格等于"暴力筛出的所有子集的降序排列"；长度必须等于 2 的置位数次方、且无重复。它专门盯住两类 bug：顺序不对（比如漏掉 0 或提前绕回）和产生重复/非法编码。

```python
rng = Random(125)

for _ in range(150):
    a = [rng.randrange(1024) for _ in range(rng.randrange(1, 16))]
    assert find_maximum_xor(a) == max((x ^ y for x in a for y in a))
```
这是随机对拍：150 组随机数组（值域 0–1023、长度 1–15），Trie 解与 O(n²) 暴力双循环逐一对照。固定种子 125 保证可复现。

```python
trie = BinaryTrie(4)

trie.insert(3)

trie.insert(3)

assert trie.max_xor(12) == 15
```
这条专测重复插入和小位宽：同一个 3 插两遍，路径共享但 size 计两次，查询 12 时仍能沿 0011 的路径全程取反位，得到 12^3=15。它顺带验证了 4 位宽度对 12（<16）的合法性检查通过。

In [6]:
assert single_number([2, 2, 1]) == 1

assert find_maximum_xor([3, 10, 5, 25, 2, 8]) == 28

for mask in range(64):
    subs = enumerate_submasks(mask)
    assert subs == sorted((s for s in range(mask + 1) if s & mask == s), reverse=True)
    assert len(subs) == len(set(subs)) == 1 << mask.bit_count()

from random import Random

rng = Random(125)

for _ in range(150):
    a = [rng.randrange(1024) for _ in range(rng.randrange(1, 16))]
    assert find_maximum_xor(a) == max((x ^ y for x in a for y in a))

trie = BinaryTrie(4)

trie.insert(3)

trie.insert(3)

assert trie.max_xor(12) == 15

print('N125: 所有本章断言通过')

N125: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 解释遍历所有mask及其submask的总次数。

**练习 2：** 扩展可回滚计数的二进制Trie。

<details>
<summary>展开思路提示（不是完整答案）</summary>

**练习 1（解释遍历所有 mask 及其 submask 的总次数）**：提示——对 n 位全集里的每个 mask，子掩码个数是 2^popcount(mask)。把所有 mask 的这个数加总：置位个数为 k 的 mask 有 C(n,k) 个，所以总数是 Σ C(n,k)·2^k = (1+2)^n = 3^n。先拿 n=2 手算验证：mask 0,1,2,3 的子掩码个数分别是 1,2,2,4，合计 9 = 3²。结论口径：这套"枚举所有集合对"的循环总代价是 3^n 量级，这就是子集 DP（比如集合覆盖问题）的标准复杂度来源。

**练习 2（扩展可回滚计数的二进制 Trie）**：方向提示——现在这棵 Trie 只能插不能删，动态题（比如"数组先加数再删数，随时问最大异或"）就无能为力。扩展思路是给每个节点（或每条边）挂一个计数器 cnt：insert 时沿路径给每个节点 cnt 加一，erase 时减一；查询时把 cnt 为 0 的节点视同不存在（相当于逻辑删除，物理节点可以留着复用）。你先手算一个小例子：4 位 Trie 插 3、3，删一个 3 之后 max_xor(12) 仍应是 15，再删一个后对 12 的查询应当只反映树中再无 3 的情形。边界要写清：删除不存在的数应报错（或按题意拒绝）。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [7]:
# 完整实现：本单元格不依赖其他单元格。
def enumerate_submasks(mask):
    if mask < 0:
        raise ValueError('nonnegative mask required')
    out = []
    sub = mask
    while True:
        out.append(sub)
        if sub == 0:
            return out
        sub = sub - 1 & mask

def single_number(nums):
    answer = 0
    for x in nums:
        answer ^= x
    return answer

class BinaryTrie:

    def __init__(self, bits=32):
        if bits < 1:
            raise ValueError('positive bit width required')
        self.bits = bits
        self.children = [[-1, -1]]
        self.size = 0

    def _check(self, x):
        if not 0 <= x < 1 << self.bits:
            raise ValueError('value outside bit width')

    def insert(self, x):
        self._check(x)
        node = 0
        for shift in range(self.bits - 1, -1, -1):
            bit = x >> shift & 1
            if self.children[node][bit] == -1:
                self.children[node][bit] = len(self.children)
                self.children.append([-1, -1])
            node = self.children[node][bit]
        self.size += 1

    def max_xor(self, x):
        self._check(x)
        if not self.size:
            raise ValueError('empty trie')
        node = answer = 0
        for shift in range(self.bits - 1, -1, -1):
            bit = x >> shift & 1
            opposite = bit ^ 1
            if self.children[node][opposite] != -1:
                answer |= 1 << shift
                node = self.children[node][opposite]
            else:
                node = self.children[node][bit]
        return answer

def find_maximum_xor(nums):
    if not nums:
        return 0
    trie = BinaryTrie(max(1, max(nums).bit_length()))
    for x in nums:
        trie.insert(x)
    return max((trie.max_xor(x) for x in nums))

# 示例与回归测试
assert single_number([2, 2, 1]) == 1

assert find_maximum_xor([3, 10, 5, 25, 2, 8]) == 28

for mask in range(64):
    subs = enumerate_submasks(mask)
    assert subs == sorted((s for s in range(mask + 1) if s & mask == s), reverse=True)
    assert len(subs) == len(set(subs)) == 1 << mask.bit_count()

from random import Random

rng = Random(125)

for _ in range(150):
    a = [rng.randrange(1024) for _ in range(rng.randrange(1, 16))]
    assert find_maximum_xor(a) == max((x ^ y for x in a for y in a))

trie = BinaryTrie(4)

trie.insert(3)

trie.insert(3)

assert trie.max_xor(12) == 15

print('N125: 所有本章断言通过')

N125: 所有本章断言通过


## 11. 代表题与迁移

- **136. Single Number**：数组里恰好一个数出现一次、其余两次，求它。single_number 即题解本体，练的是异或自消这一性质。
- **260. Single Number III**：变体——有两个落单数。这题练的是"全体异或得到 two 的异或值，再按最低不同位把数组分成两半，各自异或"的分组消去，是本章消去律的进阶用法。
- **421. Maximum XOR of Two Numbers in an Array**：数组中两数异或的最大值。find_maximum_xor 即题解本体，练的是二进制 Trie + 高位优先贪心；这题也可以用知识点清单里点名的线性基思路去对比。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。